# 02 — Generator Sanity Check
Runs CTGAN, TVAE, and CopulaGAN on a 5,000-row slice of each dataset.
Goal: confirm synthetic output is plausible before full training runs.
Checks: valid categories, sensible numeric ranges, no NaN leakage.

In [6]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import yaml
import pandas as pd
import numpy as np

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

SLICE_SIZE = 5000
SANITY_EPOCHS = 5   # small epoch count for fast iteration
SAMPLE_SIZE  = 500

print('Config loaded. Slice size:', SLICE_SIZE, '| Epochs:', SANITY_EPOCHS)

Config loaded. Slice size: 5000 | Epochs: 5


In [7]:
import os
os.getcwd()

'/Users/aryamanghaisas/SyntheticDataPipeline/notebooks'

In [9]:
import os
os.chdir('..')
print("Working directory:", os.getcwd())

Working directory: /Users/aryamanghaisas/SyntheticDataPipeline


In [10]:
from src.generators import GENERATOR_MAP

def run_sanity_check(dataset_name, generator_name, config, slice_size, epochs, sample_size):
    print(f'\n{"-"*60}')
    print(f'Dataset: {dataset_name} | Generator: {generator_name}')
    print(f'{"-"*60}')

    # Temporarily override epochs for speed
    sanity_config = yaml.safe_load(yaml.dump(config))  # deep copy
    sanity_config['generators'][generator_name]['epochs'] = epochs

    # Instantiate generator and load data
    gen_cls = GENERATOR_MAP[generator_name]
    gen = gen_cls(sanity_config, dataset_name)
    train_df = gen.load_train_data()

    # Slice
    slice_df = train_df.sample(n=min(slice_size, len(train_df)), random_state=42)
    print(f'Slice shape: {slice_df.shape}')

    # Fit
    slice_df = gen.prepare_dataframe(slice_df)
    metadata = gen.build_metadata(slice_df)
    gen.fit(slice_df, metadata)

    # Sample
    synthetic = gen.sample(sample_size)
    print(f'Synthetic shape: {synthetic.shape}')

    # --- Checks ---

    # 1. NaN check
    null_counts = synthetic.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    print(f'\n[CHECK] NaN columns: {"None" if null_cols.empty else null_cols.to_dict()}')

    # 2. Categorical validity check
    cat_cols = slice_df.select_dtypes(include='category').columns
    invalid_cats = {}
    for col in cat_cols:
        valid_vals = set(slice_df[col].dropna().unique())
        synth_vals = set(synthetic[col].dropna().unique())
        out_of_vocab = synth_vals - valid_vals
        if out_of_vocab:
            invalid_cats[col] = out_of_vocab
    print(f'[CHECK] Invalid categories: {"None" if not invalid_cats else invalid_cats}')

    # 3. Numeric range check
    num_cols = slice_df.select_dtypes(include='number').columns
    range_violations = {}
    for col in num_cols:
        real_min, real_max = slice_df[col].min(), slice_df[col].max()
        synth_min, synth_max = synthetic[col].min(), synthetic[col].max()
        margin = (real_max - real_min) * 0.2  # allow 20% margin outside real range
        if synth_min < real_min - margin or synth_max > real_max + margin:
            range_violations[col] = {
                'real': (round(real_min, 2), round(real_max, 2)),
                'synth': (round(synth_min, 2), round(synth_max, 2))
            }
    print(f'[CHECK] Range violations: {"None" if not range_violations else range_violations}')

    # 4. Target distribution comparison
    target_col = config['datasets'][dataset_name]['target_col']
    print(f'\n[CHECK] Target distribution')
    print(f'  Real:      {slice_df[target_col].value_counts(normalize=True).round(3).to_dict()}')
    print(f'  Synthetic: {synthetic[target_col].value_counts(normalize=True).round(3).to_dict()}')

    print(f'\n[PASSED] {dataset_name} × {generator_name} sanity check complete')
    return synthetic

## Diabetes 130-US

In [11]:
for gen_name in ['ctgan', 'tvae', 'copulagan']:
    run_sanity_check('diabetes_130us', gen_name, config, SLICE_SIZE, SANITY_EPOCHS, SAMPLE_SIZE)


------------------------------------------------------------
Dataset: diabetes_130us | Generator: ctgan
------------------------------------------------------------
Slice shape: (5000, 41)
PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name     Est # of Columns (CTGAN)
race                     5
gender                   2
age                      10
admission_type_id        11
discharge_disposition_id 11
admission_source_id      11
time_in_hospital         11
num_lab_procedures       11
num_procedures           11
num_medications          11
number_outpatient        11
number_emergency         11
number_inpatient         11
diag_1                   349
diag_2                   320
diag_3                   338
number_diagnoses         11
metformin                4
repaglinide              4
nateglinide              2
chlorpropamide           2
glimepiride              4
ac

/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+00.61) | Discrim. (-00.79): 100%|██████████| 5/5 [00:06<00:00,  1.34s/it]


Synthetic shape: (500, 41)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {'NO': 0.536, '>30': 0.353, '<30': 0.111}
  Synthetic: {'NO': 0.556, '>30': 0.344, '<30': 0.1}

[PASSED] diabetes_130us × ctgan sanity check complete

------------------------------------------------------------
Dataset: diabetes_130us | Generator: tvae
------------------------------------------------------------
Slice shape: (5000, 41)


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: +32.39: 100%|██████████| 5/5 [00:01<00:00,  3.30it/s]


Synthetic shape: (500, 41)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {'NO': 0.536, '>30': 0.353, '<30': 0.111}
  Synthetic: {'NO': 1.0}

[PASSED] diabetes_130us × tvae sanity check complete

------------------------------------------------------------
Dataset: diabetes_130us | Generator: copulagan
------------------------------------------------------------
Slice shape: (5000, 41)
PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name     Est # of Columns (CTGAN)
race                     5
gender                   2
age                      10
admission_type_id        11
discharge_disposition_id 11
admission_source_id      11
time_in_hospital         11
num_lab_procedures       11
num_procedures           11
num_medications          11
number_outpatient        11
number_emergency       

/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+00.86) | Discrim. (-00.72): 100%|██████████| 5/5 [00:06<00:00,  1.32s/it]

Synthetic shape: (500, 41)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {'NO': 0.536, '>30': 0.353, '<30': 0.111}
  Synthetic: {'NO': 0.636, '>30': 0.282, '<30': 0.082}

[PASSED] diabetes_130us × copulagan sanity check complete


## Home Credit

In [12]:
for gen_name in ['ctgan', 'tvae', 'copulagan']:
    run_sanity_check('home_credit', gen_name, config, SLICE_SIZE, SANITY_EPOCHS, SAMPLE_SIZE)


------------------------------------------------------------
Dataset: home_credit | Generator: ctgan
------------------------------------------------------------
Slice shape: (5000, 74)


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+01.52) | Discrim. (+00.43): 100%|██████████| 5/5 [00:05<00:00,  1.05s/it]


Synthetic shape: (500, 74)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {0: 0.92, 1: 0.08}
  Synthetic: {0: 0.97, 1: 0.03}

[PASSED] home_credit × ctgan sanity check complete

------------------------------------------------------------
Dataset: home_credit | Generator: tvae
------------------------------------------------------------
Slice shape: (5000, 74)


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: -99.99: 100%|██████████| 5/5 [00:01<00:00,  2.93it/s]


Synthetic shape: (500, 74)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {0: 0.92, 1: 0.08}
  Synthetic: {0: 1.0}

[PASSED] home_credit × tvae sanity check complete

------------------------------------------------------------
Dataset: home_credit | Generator: copulagan
------------------------------------------------------------
Slice shape: (5000, 74)


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+01.29) | Discrim. (+00.55): 100%|██████████| 5/5 [00:04<00:00,  1.05it/s]


Synthetic shape: (500, 74)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {0: 0.92, 1: 0.08}
  Synthetic: {0: 0.964, 1: 0.036}

[PASSED] home_credit × copulagan sanity check complete


In [17]:
import importlib, sys

# Remove all cached src modules
to_remove = [key for key in sys.modules if key.startswith('src')]
for key in to_remove:
    del sys.modules[key]

# Re-import fresh
from src.generators import GENERATOR_MAP
print("Reloaded. GENERATOR_MAP:", list(GENERATOR_MAP.keys()))

Reloaded. GENERATOR_MAP: ['ctgan', 'tvae', 'copulagan']


## ACS Income

In [18]:
from src.generators import GENERATOR_MAP
for gen_name in ['ctgan', 'tvae', 'copulagan']:
    run_sanity_check('acs_income', gen_name, config, SLICE_SIZE, SANITY_EPOCHS, SAMPLE_SIZE)


------------------------------------------------------------
Dataset: acs_income | Generator: ctgan
------------------------------------------------------------
Slice shape: (5000, 11)


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+02.48) | Discrim. (-00.52): 100%|██████████| 5/5 [00:02<00:00,  2.12it/s]
/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Synthetic shape: (500, 11)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {0: 0.58, 1: 0.42}
  Synthetic: {0: 0.666, 1: 0.334}

[PASSED] acs_income × ctgan sanity check complete

------------------------------------------------------------
Dataset: acs_income | Generator: tvae
------------------------------------------------------------
Slice shape: (5000, 11)


Loss: +26.66: 100%|██████████| 5/5 [00:00<00:00,  9.42it/s]
/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Synthetic shape: (500, 11)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {0: 0.58, 1: 0.42}
  Synthetic: {0: 0.998, 1: 0.002}

[PASSED] acs_income × tvae sanity check complete

------------------------------------------------------------
Dataset: acs_income | Generator: copulagan
------------------------------------------------------------
Slice shape: (5000, 11)


Gen. (+02.73) | Discrim. (-00.44): 100%|██████████| 5/5 [00:02<00:00,  1.98it/s]

Synthetic shape: (500, 11)

[CHECK] NaN columns: None
[CHECK] Invalid categories: None
[CHECK] Range violations: None

[CHECK] Target distribution
  Real:      {0: 0.58, 1: 0.42}
  Synthetic: {0: 0.614, 1: 0.386}

[PASSED] acs_income × copulagan sanity check complete


In [19]:
import pandas as pd

raw = pd.read_csv('data/raw/diabetes_130us.csv')
processed = pd.read_csv('data/processed/diabetes_130us/train.csv')

# Distribution of raw diag_1 top codes
print("Top 10 raw diag_1 codes:")
print(raw['diag_1'].replace('?', None).value_counts().head(10))

# Distribution after grouping
print("\nProcessed diag_1 chapter distribution:")
print(processed['diag_1'].value_counts())

# Cross-tab: does chapter distribution correlate with target?
print("\nChapter vs readmission (does grouping preserve signal?):")
print(pd.crosstab(
    processed['diag_1'],
    processed['readmitted'],
    normalize='index'
).round(3))

Top 10 raw diag_1 codes:
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64

Processed diag_1 chapter distribution:
diag_1
circulatory        24365
endocrine           9147
respiratory         8330
digestive           7332
symptoms            6097
genitourinary       4019
musculoskeletal     3972
external            3101
neoplasms           2794
injury              2479
infectious          2190
skin                2002
mental              1806
supplementary       1338
nervous              958
blood                883
pregnancy            554
congenital            44
external_causes        1
Name: count, dtype: int64

Chapter vs readmission (does grouping preserve signal?):
readmitted         <30    >30     NO
diag_1                              
blood            0.138  0.384  0.478
circulatory      0.114  0.357  0.529
congenital       0.023  0.227  0.750
digestive        0.105  0.355  

In [21]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    print(f"{col}: {processed[col].nunique()} unique values")

diag_1: 19 unique values
diag_2: 19 unique values
diag_3: 19 unique values


In [16]:
import inspect
from src.generators.base_generator import BaseGenerator
print(inspect.getsource(BaseGenerator.prepare_dataframe))

    def load(self, path: str) -> None:
        """Restore a trained model from disk."""

